# 03 — Polars complet

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre pourquoi Polars existe et quand le préférer à Pandas;
- manipuler un `DataFrame` Polars (strict API);
- utiliser les **expressions** (`pl.col`, `pl.when`, `pl.lit`);
- différencier **eager** et **lazy** API;
- faire des `group_by`, `join`, `window functions`;
- lire et écrire Parquet, CSV, JSON;
- exploiter l'optimiseur de requête en mode `lazy`;


## Prérequis — ce que vous devez maîtriser

Ce notebook fait partie des **Side tracks**. Il est conçu pour être autonome, à condition d'avoir suivi le niveau **Intermédiaire** de la formation principale. Vous devez donc être à l'aise avec :

- les bases de NumPy et idéalement un peu de Pandas;
- les types Python et les fonctions typées;
- la notion de table SQL et de jointure;

Ce que ce notebook **ne couvrira pas** (ou seulement en mention) :

- l'API `Series` en profondeur (miroir de Pandas);
- l'intégration avec GPU;
- la visualisation (Polars se combine très bien avec Matplotlib/Altair);


> **Note.** Polars est écrit en Rust, multi-threadé par défaut, strict sur les types, et dispose d'un vrai optimiseur de requête. Il est souvent 5 à 30 fois plus rapide que Pandas sur les opérations groupées.


## Plan

1. Installation et import
2. Créer un DataFrame
3. Inspection et types
4. L'API Expression : le cœur de Polars
5. Sélection et filtrage
6. Ajout de colonnes avec `with_columns`
7. `group_by` et agrégations
8. Jointures
9. Fenêtres : `over`
10. Eager vs Lazy
11. Optimiseur de requête
12. Lecture/écriture de fichiers
13. Interop avec Pandas/NumPy
14. Synthèse
15. Exercices


---


## 1. Installation et import

```bash
pip install polars
```


In [ ]:
import polars as pl
pl.__version__

---


## 2. Créer un DataFrame


In [ ]:
df = pl.DataFrame({
    'nom': ['Alice', 'Bob', 'Chloé', 'Dimitri'],
    'age': [30, 25, 35, 28],
    'ville': ['Paris', 'Lyon', 'Paris', 'Nice'],
    'salaire': [45000.0, 38000.0, 52000.0, 41000.0],
})
df

Polars affiche les **types** à côté des noms de colonnes. Chaque colonne a un `dtype` strict : `Int64`, `Float64`, `Utf8`, `Date`, ...


---


## 3. Inspection


In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.head(2)

In [ ]:
df.describe()

In [ ]:
df.schema

---


## 4. L'API Expression : le cœur de Polars

Contrairement à Pandas, Polars utilise des **expressions** construites avec `pl.col('nom')`. Ces expressions sont des **descriptions** d'opérations, évaluées à l'exécution dans un contexte (`select`, `with_columns`, `filter`, `group_by.agg`).


In [ ]:
pl.col('salaire') * 1.1  # expression, pas encore exécutée

In [ ]:
df.select(pl.col('salaire') * 1.1)

In [ ]:
df.select((pl.col('salaire') * 1.1).alias('salaire_majore'))

### Pourquoi ?

Une expression est **composable**, **parallélisable** et **optimisable**. Un plan peut être réorganisé automatiquement : par exemple, un filtre peut être poussé avant une projection lourde.


---


## 5. Sélection et filtrage


In [ ]:
df.select('nom', 'age')

In [ ]:
df.select(pl.col('nom'), pl.col('age'))

In [ ]:
df.filter(pl.col('age') > 28)

In [ ]:
df.filter((pl.col('age') > 28) & (pl.col('ville') == 'Paris'))

In [ ]:
df.filter(pl.col('nom').str.starts_with('A'))

---


## 6. Ajouter des colonnes avec `with_columns`


In [ ]:
df.with_columns((pl.col('salaire') * 0.3).alias('impot'))

In [ ]:
df.with_columns(
    categorie=pl.when(pl.col('salaire') > 45000)
                 .then(pl.lit('senior'))
                 .otherwise(pl.lit('junior'))
)

---


## 7. `group_by` et agrégations


In [ ]:
df.group_by('ville').agg(pl.col('salaire').mean())

In [ ]:
df.group_by('ville').agg(
    salaire_moyen=pl.col('salaire').mean(),
    age_max=pl.col('age').max(),
    n=pl.len(),
)

---


## 8. Jointures


In [ ]:
g = pl.DataFrame({'id': [1, 2, 3], 'nom': ['A', 'B', 'C']})
d = pl.DataFrame({'id': [2, 3, 4], 'score': [20, 30, 40]})

In [ ]:
g.join(d, on='id', how='inner')

In [ ]:
g.join(d, on='id', how='left')

In [ ]:
g.join(d, on='id', how='full')

---


## 9. Fenêtres : `over`


In [ ]:
df.with_columns(
    salaire_moyen_ville=pl.col('salaire').mean().over('ville'),
    rang_ville=pl.col('salaire').rank(descending=True).over('ville'),
)

`over` est l'équivalent des window functions SQL. Elle permet de calculer une agrégation tout en gardant les lignes d'origine.


---


## 10. Eager vs Lazy

L'API **eager** exécute immédiatement chaque étape (comme Pandas). L'API **lazy** construit un plan, puis l'exécute globalement via `.collect()`. Lazy est presque toujours préférable sur de gros volumes : l'optimiseur peut pousser les filtres, supprimer les colonnes inutiles, paralléliser.


In [ ]:
lf = pl.LazyFrame({
    'nom': ['Alice', 'Bob', 'Chloé'],
    'salaire': [45000, 38000, 52000],
    'ville': ['Paris', 'Lyon', 'Paris'],
})

In [ ]:
plan = (
    lf.filter(pl.col('salaire') > 40000)
      .group_by('ville')
      .agg(pl.col('salaire').mean())
)
plan

In [ ]:
plan.collect()

---


## 11. Optimiseur de requête


In [ ]:
plan.explain()

`explain` affiche le plan optimisé. Vous verrez parfois `PROJECT */4 COLUMNS` ou des prédicats poussés vers le bas (predicate pushdown).


---


## 12. Lecture/écriture de fichiers


In [ ]:
# pl.read_csv('fichier.csv', separator=';')

In [ ]:
# pl.read_parquet('fichier.parquet')

In [ ]:
# pl.scan_csv('gros.csv').filter(pl.col('x') > 0).collect()  # lazy !

In [ ]:
# df.write_parquet('out.parquet')

**`scan_*`** construit un LazyFrame : la lecture réelle ne commence qu'au `collect()`, ce qui permet de lire seulement les colonnes et lignes utiles. Gain énorme sur Parquet.


---


## 13. Interop avec Pandas / NumPy


In [ ]:
pdf = df.to_pandas()
pdf

In [ ]:
df_from_pandas = pl.from_pandas(pdf)
df_from_pandas

In [ ]:
df.select('age').to_numpy()

---


## Synthèse

| Notion | À retenir |
|---|---|
| `DataFrame` | strict sur les types, multi-threadé, pas d'index |
| Expression | description composable d'une opération (`pl.col`, `pl.when`) |
| Contexte | `select`, `filter`, `with_columns`, `group_by.agg`, `sort` |
| `LazyFrame` | plan non exécuté → `collect()` pour lancer |
| `scan_*` | équivalent lazy de `read_*`, permet le pushdown |
| `over` | window function SQL-like |
| Parquet | format privilégié, colonnaire, très rapide |


### Règles à retenir

- **Préférer Lazy** dès que le traitement est non-trivial.
- **Utiliser `scan_parquet`** sur de gros volumes — lecture minimale.
- **Pas d'index** contrairement à Pandas — c'est voulu et plus sûr.
- **Les expressions sont composables** : construisez par morceaux.
- **Polars est strict** : pas de conversion implicite int/float.


---


## Exercices

Pour chaque exercice : écrivez une **fonction typée**, testez-la, puis comparez avec la correction. N'ouvrez pas la correction avant d'avoir vraiment cherché.


### Exercice 1 — Filtrer et agréger

Écrivez `ca_par_categorie(df)` qui renvoie le CA total par catégorie, trié décroissant.


In [ ]:
import polars as pl

def ca_par_categorie(df: pl.DataFrame) -> pl.DataFrame:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
import polars as pl

def ca_par_categorie(df: pl.DataFrame) -> pl.DataFrame:
    return (df.group_by('categorie')
              .agg(ca=pl.col('montant').sum())
              .sort('ca', descending=True))
```

</details>


### Exercice 2 — Colonne dérivée

Ajoutez une colonne `tranche` valant `'jeune'` si age < 30, sinon `'senior'`.


In [ ]:
def ajouter_tranche(df: pl.DataFrame) -> pl.DataFrame:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
def ajouter_tranche(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        tranche=pl.when(pl.col('age') < 30).then(pl.lit('jeune')).otherwise(pl.lit('senior'))
    )
```

</details>


### Exercice 3 — Lazy pipeline

Écrivez `ville_ayant_salaire_median_max(lf)` qui renvoie, via une requête lazy, le nom de la ville dont le salaire médian est le plus élevé.


In [ ]:
def ville_ayant_salaire_median_max(lf: pl.LazyFrame) -> str:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
def ville_ayant_salaire_median_max(lf: pl.LazyFrame) -> str:
    df = (lf.group_by('ville')
            .agg(m=pl.col('salaire').median())
            .sort('m', descending=True)
            .head(1)
            .collect())
    return df['ville'][0]
```

</details>


### Exercice 4 — Jointure

Joignez `clients(id, nom)` et `commandes(client_id, total)` et renvoyez le CA total par client.


In [ ]:
def ca_par_client(clients: pl.DataFrame, commandes: pl.DataFrame) -> pl.DataFrame:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
def ca_par_client(clients: pl.DataFrame, commandes: pl.DataFrame) -> pl.DataFrame:
    ca = commandes.group_by('client_id').agg(ca=pl.col('total').sum())
    return clients.join(ca, left_on='id', right_on='client_id', how='left').fill_null(0)
```

</details>


---


## Ressources externes

- [Documentation Polars](https://docs.pola.rs/)
- [Polars User Guide](https://docs.pola.rs/user-guide/)
- [Modern Polars](https://kevinheavey.github.io/modern-polars/)
- [Polars vs Pandas](https://pola.rs/posts/benchmarks/)
